# ch16 练习：时间序列处理

与讲解版逐 Cell 对应，`assert` 验收保留。卡住回讲解版看对应小节。

## 一、考点回顾

| 竞赛评分点 | 本节覆盖 |
|---|---|
| 数据准备及处理 | 重采样 + 日频特征 + 滞后特征 |
| 数据探索 | 字符串切片 + 节假日对比 |

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.simplefilter("ignore")
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

DATA = Path("data")

load = pd.read_csv(DATA / "load_curve.csv")
load["时间戳"] = pd.to_datetime(load["时间戳"])

KEY = "台区编号"
VAL = "负荷值"
TS = "时间戳"

# 单台区小时级序列（DatetimeIndex）
st = load[load[KEY] == "STATION_A_01"].sort_values(TS).set_index(TS)

print("pandas", pd.__version__)
print("load", load.shape, "| 台区", load[KEY].nunique(),
      "| 时间范围", load[TS].min().date(), "~", load[TS].max().date())
print("st", st.shape, "| 负荷值 NaN", int(st[VAL].isna().sum()))

In [ ]:
# 脚手架：写在 @@todo 之外，练习版原样保留
def probe(fn, *args, **kwargs):
    """把「调用可能抛异常」写成返回值，避免在挖空块里写 try/except。"""
    try:
        return "ok", fn(*args, **kwargs)
    except Exception as exc:
        return "err", type(exc).__name__


print("st 索引类型:", type(st.index).__name__, "| 频率:", st.index.freq)

## 题 1：resample 频率与标签（对应讲解 §3.2）

In [ ]:
# TODO(1)：对 st 按周重采样取负荷均值，存入 w；再存 label="left" 版 w_left
# 提示：st.resample("W")[VAL].mean()
w = ____
w_left = ____

print("W 桶数:", len(w), "| 默认首桶:", w.index[0].date(),
      "| left 首桶:", w_left.index[0].date())
assert len(w) == 14
assert str(w.index[0].date()) == "2026-01-04"
assert str(w_left.index[0].date()) == "2025-12-28"

## 题 2：NaN 三分家（对应讲解 §3.3）

In [ ]:
# TODO(2)：对 st 按小时重采样：桶数存入 n_buckets、空桶数存入 n_empty、
#            负荷均值 NaN 桶数存入 n_nan
hr = ____
n_buckets = ____
n_empty = ____
n_nan = ____

# TODO(3)：对 st 按天聚合 mean/count/max/min 存入 daily；最小 count 存入 min_cnt
daily = ____
min_cnt = ____

print("小时桶:", n_buckets, "| 空桶:", n_empty, "| NaN 桶:", n_nan)
print("日均表:", daily.shape, "| 最小 count:", min_cnt)
assert n_buckets == 2160 and n_empty == 0 and n_nan == 23
assert daily.shape == (90, 4) and min_cnt == 18

## 题 3：pd.Grouper 分组重采样（对应讲解 §3.4）

In [ ]:
# TODO(4)：全表按 [台区, Grouper(天)] 求负荷均值存入 gd；unstack 成宽表存入 wd
# 提示：pd.Grouper(key=TS, freq="D")
gd = ____
wd = ____

print("gd:", gd.shape, gd.index.names, "| wd:", wd.shape)
assert gd.shape == (270,) and gd.index.names == [KEY, TS]
assert wd.shape == (90, 3)
assert round(float(gd.iloc[0]), 4) == 484.035

## 题 4：日频特征四件套（对应讲解 §3.5）

In [ ]:
# TODO(5)：给 feat 加四列：月份 / 小时 / 星期 / 是否周末（周末=1）
# 提示：feat.index.month ；(feat.index.dayofweek >= 5).astype(int)
feat = ____
feat["月份"] = ____
feat["小时"] = ____
feat["星期"] = ____
feat["是否周末"] = ____

print("月份:", sorted(feat["月份"].unique().tolist()),
      "| 周末行数:", int(feat["是否周末"].sum()))
assert sorted(feat["月份"].unique().tolist()) == [1, 2, 3]
assert int(feat["是否周末"].sum()) == 624
assert int(feat["星期"].value_counts()[2]) == 288

## 题 5：组内滞后（对应讲解 §3.6）

In [ ]:
# TODO(6)：按 [台区, 时间戳] 排序后，组内 shift(24) 生成「滞后1天」、
#            diff(24) 生成「日环比」；再生成不分组版「不分组滞后」
g = ____
g["滞后1天"] = ____
g["日环比"] = ____
g["不分组滞后"] = ____

print("滞后1天 NaN:", int(g["滞后1天"].isna().sum()),
      "| 日环比 NaN:", int(g["日环比"].isna().sum()))
print("错位行数:", int((g["滞后1天"].isna() != g["不分组滞后"].isna()).sum()))
print("验收 - 各组前 24 行全 NaN:",
      bool(g.groupby(KEY)["滞后1天"].head(24).isna().all()))
assert int(g["滞后1天"].isna().sum()) == 120
assert int(g["日环比"].isna().sum()) == 168
assert int((g["滞后1天"].isna() != g["不分组滞后"].isna()).sum()) == 48

## 题 6（综合）：字符串切片 + 节假日对比（对应讲解 §3.7-3.8）

In [ ]:
# TODO(7)：用字符串切片取 A_01 的 1 月整月存入 jan、单日 2026-01-01 存入 day1
jan = ____
day1 = ____

# TODO(8)：全表按是否节假日分两组算负荷均值：hol_mean / work_mean
# 提示：load.loc[load["是否节假日"], VAL].mean()
hol_mean = ____
work_mean = ____

print("1 月:", jan.shape, "| 单日:", day1.shape,
      "| 单日均值:", round(float(day1[VAL].mean()), 3))
print("节假日:", round(float(hol_mean), 3), "vs 工作日:", round(float(work_mean), 3))
assert jan.shape == (744, 4) and day1.shape == (24, 4)
assert round(float(day1[VAL].mean()), 3) == 484.035
assert round(float(hol_mean), 3) == 513.004
assert round(float(work_mean), 3) == 678.191

---

## 综合自查

1. 题 1 两种标签各适合什么报表场景？
2. 题 2 的 23 个 NaN 桶是缺口吗？用什么列证明？
3. 题 5 的 120 和 48 分别怎么算出来的？
4. 滞后特征的验收标准是什么？为什么重要？

全部答得上来，进入 ch17（性能与内存）。